# 🧠 Notebook 3: Regresión Logística (Probabilidades)

En el notebook anterior observamos que el **Perceptrón** tiene limitaciones:
1.  Es un clasificador binario rígido (sí/no).
2.  No maneja bien datos que no son linealmente separables.
3.  No nos dice "cuán seguro" está de su decisión.

En este laboratorio implementaremos la **Regresión Logística**. A pesar de su nombre, es un algoritmo de clasificación que utiliza la función **Sigmoide** para estimar la probabilidad de que un dibujo pertenezca a una clase.

### Objetivo
Sustituir la decisión brusca del Perceptrón por una **estimación de probabilidad** ($P(y=1|x)$) para mejorar la robustez del sistema de reconocimiento de *Doodles*.

Ejecutar los siguientes comandos para importar las librerías necesarias y configurar el entorno de trabajo.

In [32]:
!pip install seaborn scikit-learn pandas pillow gradio matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\hecto\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


<p style="page-break-after:always;"></p>

# Regresión logística

In [33]:
import numpy as np

**OneHotEncoding:** $\;$ función que convierte las etiquetas de clase a representación *one-hot*

$$\operatorname{one-hot}(y)%
=\boldsymbol{y}%
=\begin{pmatrix}y_1\\\vdots\\y_C\end{pmatrix}%
=\begin{pmatrix}\mathbb{I}(y=1)\\\vdots\\\mathbb{I}(y=C)\end{pmatrix}\in\{0,1\}^C%
\quad\text{con}\quad%
\sum_c y_c=1$$

$\;$ donde $y$ es una variable categórica que toma un valor entre $C$ categorías posibles, $\,\{1,\dotsc,C\}$

In [34]:
def OneHotEncoding(y):
  # Etiquetas de clase únicas
  c = np.unique(y); C = c.size;

  # Mapeamos las etiquetas de clase de 0 a C-1
  # Creamos una correspondencia de los valores de c a sus índices correspondientes
  mapping = {value: idx for idx, value in enumerate(c)}
  # Convertimos y usando esta correspondencia
  y = np.vectorize(mapping.get)(y)

  # Generamos la representación "one-hot" usando el método np.eye
  return np.eye(C)[y]

**ComputeGradient:** $\;$ actualización de pesos para regresión logística

Gradiente descendente aplicado a la regresión logística:

Neg-log-likelihood (NLL): $\;$ la neg-log-verosimilitud es una función objetiva convexa
$$\operatorname{NLL}(\mathbf{W})%
=\frac{1}{N}\sum_{n=1}^N-\log p(\boldsymbol{y}_n\mid\boldsymbol{\mu}_n)
\qquad\text{con}\quad%
\boldsymbol{\mu}_n=\mathcal{S}(\boldsymbol{a}_n)%
\quad\text{y}\quad%
\boldsymbol{a}_n=\mathbf{W}^t\boldsymbol{x}_n$$

Gradiente NLL:
$$\begin{pmatrix}%
\frac{\partial\operatorname{NLL}}{\partial W_{11}}&\cdots&\frac{\partial\operatorname{NLL}}{\partial W_{1C}}\\%
\vdots&\ddots&\vdots\\%
\frac{\partial\operatorname{NLL}}{\partial W_{D1}}&\cdots&\frac{\partial\operatorname{NLL}}{\partial W_{DC}}\\%
\end{pmatrix}
=\frac{\partial\operatorname{NLL}}{\partial\mathbf{W}^t}%
=\frac{1}{N}\sum_{n=1}^N\frac{\partial(-\log p(\boldsymbol{y}_n\mid\boldsymbol{\mu}_n))}{\partial \mathbf{W}^t}%
=\frac{1}{N}\sum_{n=1}^N\boldsymbol{x}_n(\boldsymbol{\mu}_n-\boldsymbol{y}_n)^t$$

In [35]:
def ComputeGradient(X, Y, W):
  N, D = X.shape
  X = np.concatenate((np.ones((N,1)), X),axis=1)
  N, C = Y.shape
  Z = np.zeros((N, C)).astype(float)

  # Calculamos los logits
  a = X @ W
  # Normalizamos los logits para una computación robusta que evite desbordamientos ("overflows")
  a -= a.max(axis=1, keepdims=True)
  # Calculamos la softmax (etiqueta de clase probabilística)
  mu = np.exp(a)
  mu /= np.sum(mu, axis=1, keepdims=True)
  # Restamos la etiqueta actual a la etiqueta de clase probabilística
  Z = mu - Y
  # El gradiente es el sumatorio, para todas las muestras, del producto escalar externo
  return (X.T @ Z)/N

<p style="page-break-after:always;"></p>

**LogisticRegressionClassification:** $\;$ clasificación de las muestras de acuerdo con una matriz de pesos dada

$$c(\boldsymbol{x})%
=\operatorname*{argmax}\limits_c\;\boldsymbol{\mu}_c%
\quad\text{con}\quad%
\boldsymbol{\mu}_c=\mathcal{S}(\boldsymbol{a})_c,\quad%
\boldsymbol{a}=f(\boldsymbol{x};\mathbf{W})=\mathbf{W}^t\boldsymbol{x},\quad%
\mathbf{W}\in\mathbb{R}^{D\times C}\quad\text{y}\quad\boldsymbol{x}\in\mathbb{R}^D$$

In [36]:
def LogisticRegressionClassification(X, W):
  N, D = X.shape
  X = np.concatenate((np.ones((N,1)), X),axis=1)

  # Calculamos los logits
  a = X @ W
  # Normalizamos los logits para una computación robusta que evite desbordamientos ("overflows")
  a -= a.max(axis=1, keepdims=True)
  # Calculamos la softmax (etiqueta de clase probabilística)
  mu = np.exp(a)
  mu /= np.sum(mu, axis=1, keepdims=True)
  # La clase es el argumento del máximo de la etiqueta de clase probabilística
  return np.argmax(mu,axis=1)

**LogisticRegressionTraining:** $\;$ implementación de entrenamiento por lotes (*batch training*) basada en gradiente descendente estocástico

Gradiente descendente aplicado a la regresión logística: $\displaystyle\quad\mathbf{W}_0=\mathbf{0};\quad\mathbf{W}_{i+1}=\mathbf{W}_i -\eta_i\frac{\partial\operatorname{NLL}}{\partial\mathbf{W}^t}\biggr\vert_{\mathbf{W}_i}\quad i=0,1,\ldots$

In [37]:
def LogisticRegressionTraining(X, y, W=None, bs=1, maxEpochs=10, eta=1e-2, tol=1e-3):
  rng = np.random.default_rng(seed=23)
  N, D = X.shape;
  Y = OneHotEncoding(y)
  N, C = Y.shape
  if W is None:  # W_0 = 0
    W = np.zeros((1+D, C))
  grad = np.inf; epoch = 0
  while np.max(np.abs(grad)) > tol and epoch < maxEpochs:  # W_i+1 = W_i - eta_i * grad_W_i
    perm = rng.permutation(N); Xperm = X[perm]; Yperm = Y[perm]
    j = 0
    for j in range(N//bs):
      X_batch = Xperm[j*bs:(j+1)*bs]
      Y_batch = Yperm[j*bs:(j+1)*bs]
      grad = ComputeGradient(X_batch, Y_batch, W)
      W = W - eta*grad
    # Si hay muestras restantes después de la partición en lotes
    if((j+1)*bs < N):
      X_batch = Xperm[(j+1)*bs:]
      Y_batch = Yperm[(j+1)*bs:]
      grad = ComputeGradient(X_batch, Y_batch, W)
      W = W - eta*grad
    epoch += 1
  return W


# Ejercicio: Regresión Logística aplicada a MyDoodles

Leemos y hacemos la partición del conjunto de datos. A continuación, ajustamos el número máximo de épocas de entrenamiento para ver cómo evoluciona el error.

**Leemos y hacemos la partición del conjunto de datos:**

In [38]:
import numpy as np
from sklearn.model_selection import train_test_split
import os

# Rutas a tus archivos (Asegúrate de que la ruta sea correcta)
path_X = '../data/doodles_X.npy'
path_y = '../data/doodles_y.npy'

# Lectura de datos estilo "profesor"
with open(path_X, 'rb') as fd:
    X = np.load(fd)

with open(path_y, 'rb') as fd:
    y = np.load(fd).astype(int) # Aseguramos que sean enteros

# Partición del dataset (Random State 23 como en la imagen)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    random_state=23
)

# Definimos N y M como pide el ejercicio
N = len(X_train)
M = len(X_test)

print(f"✅ Datos cargados: N={N} (Train), M={M} (Test)")

✅ Datos cargados: N=80 (Train), M=20 (Test)


**Ejercicio: Ajustamos el número máximo de épocas de entrenamiento:**

In [39]:
print(f"{'bs':>4} {'maxEps':>8} {'eta':>8} {'trainErr':>10} {'testErr':>10}")
print("-" * 45)

# Configuración inicial del ejercicio
bs = 10
eta = 1e-2

# Bucle para probar distintas épocas
for maxEpochs in (5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000):

    # 1. Entrenar
    W = LogisticRegressionTraining(X_train, y_train, bs=bs, maxEpochs=maxEpochs, eta=eta)

    # 2. Calcular error en Entrenamiento
    haty_train = LogisticRegressionClassification(X_train, W)
    acc_train = np.sum(haty_train == y_train) / N

    # 3. Calcular error en Test
    haty_test = LogisticRegressionClassification(X_test, W)
    acc_test = np.sum(haty_test == y_test) / M

    # 4. Imprimir resultados (Error = 1.0 - Accuracy)
    print(f"{bs:4d} {maxEpochs:8d} {eta:.1e} {1.0-acc_train:9.1%} {1.0-acc_test:9.1%}")

  bs   maxEps      eta   trainErr    testErr
---------------------------------------------
  10        5 1.0e-02     25.0%     70.0%
  10       10 1.0e-02     15.0%     65.0%
  10       20 1.0e-02     13.7%     65.0%
  10       50 1.0e-02      2.5%     55.0%
  10      100 1.0e-02      0.0%     35.0%
  10      200 1.0e-02      0.0%     30.0%
  10      500 1.0e-02      0.0%     20.0%
  10     1000 1.0e-02      0.0%     20.0%
  10     2000 1.0e-02      0.0%     20.0%
  10     5000 1.0e-02      0.0%     20.0%


**Ejercicio: Ajustamos la tasa de aprendizaje o *learning rate* ($\eta$/eta):**

In [40]:
print(f"{'bs':>4} {'maxEps':>8} {'eta':>8} {'trainErr':>10} {'testErr':>10}")
print("-" * 45)

# Configuración fija para este experimento
bs = 10
maxEpochs = 500

# Probamos potencias de 10 para eta
for eta in (1e-4, 1e-3, 1e-2, 1e-1, 1, 1e1, 1e2):

    # 1. Entrenar
    W = LogisticRegressionTraining(X_train, y_train, bs=bs, maxEpochs=maxEpochs, eta=eta)

    # 2. Validar (Train)
    haty_train = LogisticRegressionClassification(X_train, W)
    acc_train = np.sum(haty_train == y_train) / N

    # 3. Validar (Test)
    haty_test = LogisticRegressionClassification(X_test, W)
    acc_test = np.sum(haty_test == y_test) / M

    # 4. Imprimir
    print(f"{bs:4d} {maxEpochs:8d} {eta:.1e} {1.0-acc_train:9.1%} {1.0-acc_test:9.1%}")

  bs   maxEps      eta   trainErr    testErr
---------------------------------------------
  10      500 1.0e-04     25.0%     70.0%
  10      500 1.0e-03      2.5%     55.0%
  10      500 1.0e-02      0.0%     20.0%
  10      500 1.0e-01      0.0%     20.0%
  10      500 1.0e+00      0.0%     25.0%
  10      500 1.0e+01      0.0%     30.0%
  10      500 1.0e+02      2.5%     70.0%


**Ejerc.: Ajustamos tamaño de lote (*batch size*):** parámetro *bs* define cada cuántas muestras se actualiza la matriz de pesos

In [41]:
print(f"{'bs':>4} {'maxEps':>8} {'eta':>8} {'trainErr':>10} {'testErr':>10}")
print("-" * 45)

# Usamos los mejores valores encontrados antes
maxEpochs = 500
eta = 1.0e-01  # CORRECCIÓN: En la imagen pone +02, pero usamos -02 que funcionaba mejor

# Probamos diferentes tamaños de lote
for bs in (1, 2, 5, 10, 20, 50, 100):

    # 1. Entrenar
    W = LogisticRegressionTraining(X_train, y_train, bs=bs, maxEpochs=maxEpochs, eta=eta)

    # 2. Validar (Train)
    haty_train = LogisticRegressionClassification(X_train, W)
    acc_train = np.sum(haty_train == y_train) / N

    # 3. Validar (Test)
    haty_test = LogisticRegressionClassification(X_test, W)
    acc_test = np.sum(haty_test == y_test) / M

    # 4. Imprimir
    print(f"{bs:4d} {maxEpochs:8d} {eta:.1e} {1.0-acc_train:9.1%} {1.0-acc_test:9.1%}")

  bs   maxEps      eta   trainErr    testErr
---------------------------------------------
   1      500 1.0e-01      0.0%     20.0%
   2      500 1.0e-01      0.0%     20.0%
   5      500 1.0e-01      0.0%     20.0%
  10      500 1.0e-01      0.0%     20.0%
  20      500 1.0e-01      0.0%     20.0%
  50      500 1.0e-01      0.0%     20.0%
 100      500 1.0e-01     88.8%     95.0%


**Ejercicio: Classificador final:** $\;$ Entrenamos el clasificador final con todos los datos disponibles y con los mejores parámetros observados. Después lo guardamos, y lo cargamos para evaluarlo

In [42]:
# Ejercicio: Clasificador final
print("🚀 Entrenando modelo definitivo con TODOS los datos...")

# 1. Configuración GANADORA
# (Sustituye estos valores si en tu tabla anterior viste mejores números)
bs = 10
maxEpochs = 500
eta = 1.0e-01  # 0.1

# 2. Entrenamos usando X e y (el dataset completo, sin dividir)
W_final = LogisticRegressionTraining(X, y, bs=bs, maxEpochs=maxEpochs, eta=eta)

# 3. Guardamos los pesos
# Guardamos en la carpeta models para mantener el orden
if not os.path.exists('../models'):
    os.makedirs('../models')

ruta_modelo = '../models/logistic_weights.npy'
np.save(ruta_modelo, W_final)

print(f"💾 Modelo guardado correctamente en: {ruta_modelo}")

🚀 Entrenando modelo definitivo con TODOS los datos...
💾 Modelo guardado correctamente en: ../models/logistic_weights.npy


In [43]:
print("📂 Verificando carga del modelo...")

ruta_modelo = '../models/logistic_weights.npy'

# 1. Cargar el archivo recién creado
with open(ruta_modelo, 'rb') as fd:
    W_loaded = np.load(fd)

# 2. Comprobación rápida con el conjunto de Test
# (Aunque hemos entrenado con todo, usamos Test para ver que la matriz funciona)
haty_test = LogisticRegressionClassification(X_test, W_loaded)
accuracy = np.sum(haty_test == y_test) / y_test.size

print(f"✅ Modelo cargado y funcional.")
print(f"📊 Error de evaluación del clasificador final: {1.0 - accuracy:.1%}")

📂 Verificando carga del modelo...
✅ Modelo cargado y funcional.
📊 Error de evaluación del clasificador final: 0.0%


# 🎨 Clasifica tus propios Doodles (Regresión Logística)

En esta sección final, probaremos el modelo en el mundo real.

A diferencia del script original del curso (que depende de archivos externos para Google Colab), hemos implementado aquí una **interfaz completa con Gradio**.

El sistema:
1.  Cargará tu matriz de pesos óptima ($W$) entrenada con Regresión Logística.
2.  Procesará tu dibujo aplicando un filtro de engrosamiento para evitar perder detalles al reducir a 16x16.
3.  Utilizará la función **Softmax** para calcular la probabilidad de cada clase y elegir la ganadora.

In [44]:
import gradio as gr
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import os

# --- 1. CONFIGURACIÓN ---
MODEL_PATH = '../models/logistic_weights.npy'
DOODLE_MAP = {
    0: "🙂 Carita Feliz", 1: "❤️ Corazón", 2: "⭐ Estrella", 3: "⚡ Rayo",
    4: "🌙 Luna",        5: "☁️ Nube",    6: "🌲 Árbol",   7: "🏠 Casa",
    8: "☹️ Carita Triste", 9: "☀️ Sol"
}

# --- 2. CARGA DEL MODELO ---
if os.path.exists(MODEL_PATH):
    with open(MODEL_PATH, 'rb') as fd:
        W_loaded = np.load(fd)
    print(f"✅ Modelo de Regresión Logística cargado.")
else:
    raise FileNotFoundError("❌ No se encuentra el archivo .npy. ¡Ejecuta el entrenamiento arriba!")

# --- 3. LÓGICA DE PREDICCIÓN (CON PREPROCESAMIENTO ROBUSTO) ---
def predict_logistic(image_input):
    if image_input is None: return "⚠️ Dibuja algo primero."

    # A. Gestión de entrada de Gradio
    image = image_input.get("composite") if isinstance(image_input, dict) else image_input

    # B. Preprocesamiento (Truco del 'Rotulador Gordo')
    img = image.convert("L")
    img = ImageOps.invert(img)
    # Engrosamos el trazo para que no desaparezca al reducir
    img = img.filter(ImageFilter.MaxFilter(9))
    img = img.resize((16, 16), resample=Image.BILINEAR)

    # C. Preparar datos para el modelo
    # Convertimos a array y aplanamos -> Forma (1, 256)
    x = np.array(img)
    x_flat = x.flatten().reshape(1, -1)

    # D. Inferencia (Matemáticas de Regresión Logística)
    # Usamos la función que ya definiste 'LogisticRegressionClassification'
    # Si te da error de que no existe, descomenta las líneas de abajo para hacerlo manual:
    """
    N, D = x_flat.shape
    X_aug = np.concatenate((np.ones((N,1)), x_flat), axis=1) # Añadir bias
    logits = X_aug @ W_loaded
    logits -= logits.max(axis=1, keepdims=True) # Estabilidad
    probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
    pred_idx = np.argmax(probs, axis=1)[0]
    confidence = probs[0][pred_idx]
    """
    # Llamada directa a tu función (asegúrate de haber ejecutado la celda de funciones arriba)
    pred_idx = LogisticRegressionClassification(x_flat, W_loaded)[0]

    return f"🤖 Predicción: {DOODLE_MAP.get(pred_idx, 'Desconocido')}"

# --- 4. LANZAR INTERFAZ ---
interface = gr.Interface(
    fn=predict_logistic,
    inputs=gr.Sketchpad(label="Dibuja aquí (Trazo lento y claro)", type="pil", brush=gr.Brush(default_size=20)),
    outputs="text",
    title="🧠 Juez de Doodles (Regresión Logística)",
    description="Dibuja y deja que el modelo de probabilidades decida qué es."
)

interface.launch(debug=True)

✅ Modelo de Regresión Logística cargado.
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
